# PDF Table Extraction Testing

This notebook tests PDF extraction with intelligent table detection:
- **Gemini 2.5 Flash-Lite** for vision-based table detection
- **Tabula** for structured table extraction
- **PyMuPDF** for text extraction outside table regions
- **Word count validation** (90% threshold) with OCR fallback
- **Scanned PDF detection** to skip non-digital documents

**NO Pinecone** - Just extraction and validation testing.

### 1. Setup and Imports

In [ ]:
import os
import io
import json
from pathlib import Path
from typing import List, Dict, Optional
from dotenv import load_dotenv

# Data Handling & Processing
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
from tabula import read_pdf  # Correct import from tabula-py

# Google Generative AI
import google.generativeai as genai

# Load environment variables
load_dotenv()

# Configure Gemini API
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment variables")

genai.configure(api_key=GEMINI_API_KEY)
print("✓ API configured successfully")

### 2. Helper Functions

In [ ]:
# =============================================================================
# SCANNED PDF DETECTION
# =============================================================================

def is_scanned_pdf(file_path: str, sample_pages: int = 3, threshold: int = 100) -> bool:
    """
    Detect if a PDF is scanned by checking text content on sample pages.
    
    Args:
        file_path: Path to PDF file
        sample_pages: Number of pages to sample (default: 3)
        threshold: Average characters per page threshold (default: 100)
        
    Returns:
        True if PDF appears to be scanned, False otherwise
    """
    try:
        doc = fitz.open(file_path)
        total_chars = 0
        pages_to_check = min(sample_pages, len(doc))
        
        if pages_to_check == 0:
            return True
        
        for i in range(pages_to_check):
            page = doc[i]
            text = page.get_text().strip()
            total_chars += len(text)
        
        doc.close()
        avg_chars_per_page = total_chars / pages_to_check
        
        is_scanned = avg_chars_per_page < threshold
        status = "SCANNED" if is_scanned else "DIGITAL"
        print(f"   PDF Detection: {status} (avg {avg_chars_per_page:.0f} chars/page)")
        
        return is_scanned
        
    except Exception as e:
        print(f"   Error detecting PDF type: {e}")
        return False


# =============================================================================
# COORDINATE CONVERSION HELPER
# =============================================================================

def convert_pixel_to_pdf_coords(pixel_bbox: List[float], img_width: int, img_height: int, 
                                 pdf_width: float, pdf_height: float) -> List[float]:
    """
    Convert pixel coordinates from image to PDF points.
    
    Args:
        pixel_bbox: [x1, y1, x2, y2] in pixels
        img_width: Image width in pixels
        img_height: Image height in pixels
        pdf_width: PDF width in points
        pdf_height: PDF height in points
        
    Returns:
        [x1, y1, x2, y2] in PDF points
    """
    x1_px, y1_px, x2_px, y2_px = pixel_bbox
    
    # Calculate scale factors
    x_scale = pdf_width / img_width
    y_scale = pdf_height / img_height
    
    # Convert to PDF points
    x1_pdf = x1_px * x_scale
    y1_pdf = y1_px * y_scale
    x2_pdf = x2_px * x_scale
    y2_pdf = y2_px * y_scale
    
    return [x1_pdf, y1_pdf, x2_pdf, y2_pdf]


# =============================================================================
# GEMINI VISION TABLE DETECTION
# =============================================================================

def detect_tables_with_gemini(page_image: Image.Image, page_width: float, 
                               page_height: float, page_num: int) -> Dict:
    """
    Use Gemini 2.0 Flash-Lite to detect tables in a page image.
    
    Args:
        page_image: PIL Image of the page
        page_width: Page width in PDF points
        page_height: Page height in PDF points
        page_num: Page number for logging
        
    Returns:
        Dictionary with table detection results
    """
    
    # Get image dimensions
    img_width, img_height = page_image.size
    
    # Debug output
    print(f"   Debug: Image={img_width}×{img_height}px, PDF={page_width:.1f}×{page_height:.1f}pts")
    
    prompt = f"""You are a document analysis system. Analyze this page image and detect ALL tables (both bordered and borderless).

**IMAGE INFORMATION:**
- This image is {img_width} pixels wide × {img_height} pixels tall
- You must return bounding boxes in PIXEL coordinates based on what you see in the image

**ONE-SHOT EXAMPLE:**
If you see a table in the image:
- Top-left corner at pixel position (100, 150)  
- Bottom-right corner at pixel position (500, 400)
- You should return: {{"id": 1, "bbox": [100, 150, 500, 400], "type": "bordered"}}

**CRITICAL RULES:**
1. Return coordinates in PIXELS (not points!)
2. Bounding box format: [left, top, right, bottom] in pixels
3. Coordinates must be within image bounds:
   - x values: 0 to {img_width}
   - y values: 0 to {img_height}
4. Include tables with borders, borderless tables, and tables with only horizontal lines
5. Do NOT include headers/footers unless they are actual tables

**OUTPUT FORMAT:**
Return ONLY a valid JSON object, no other text or markdown.

**JSON Schema:**
{{
  "tables": [
    {{
      "id": 1,
      "bbox": [x1_pixels, y1_pixels, x2_pixels, y2_pixels],
      "type": "bordered" | "borderless"
    }}
  ],
  "page_has_tables": true | false
}}

Analyze the image and return JSON:"""
    
    try:
        vision_model = genai.GenerativeModel(model_name="gemini-2.0-flash-lite")
        response = vision_model.generate_content([prompt, page_image])
        response_text = response.text.strip()
        
        # Extract JSON from response
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
        
        result = json.loads(response_text)
        
        # Validate structure
        if "tables" not in result or "page_has_tables" not in result:
            print(f"   ⚠ Page {page_num}: Invalid response structure")
            return {"tables": [], "page_has_tables": False, "error": "invalid_structure"}
        
        # Convert pixel coordinates to PDF points and validate
        validated_tables = []
        for table in result["tables"]:
            pixel_bbox = table.get("bbox", [])
            if len(pixel_bbox) == 4:
                # First validate pixel coordinates
                x1_px, y1_px, x2_px, y2_px = pixel_bbox
                
                if not (0 <= x1_px < x2_px <= img_width and 0 <= y1_px < y2_px <= img_height):
                    print(f"   ⚠ Page {page_num}: Invalid pixel bbox {pixel_bbox}, skipping")
                    continue
                
                # Convert to PDF points
                pdf_bbox = convert_pixel_to_pdf_coords(
                    pixel_bbox, img_width, img_height, page_width, page_height
                )
                
                x1, y1, x2, y2 = pdf_bbox
                
                # Validate PDF coordinates
                if (0 <= x1 < x2 <= page_width and 0 <= y1 < y2 <= page_height):
                    table["bbox"] = pdf_bbox  # Store converted coordinates
                    table["pixel_bbox"] = pixel_bbox  # Keep original for debugging
                    validated_tables.append(table)
                    print(f"   ✓ Table {table['id']}: pixels{pixel_bbox} → PDF{[f'{c:.1f}' for c in pdf_bbox]}")
                else:
                    print(f"   ⚠ Page {page_num}: Invalid PDF bbox {pdf_bbox} (from pixels {pixel_bbox}), skipping")
        
        result["tables"] = validated_tables
        result["page_has_tables"] = len(validated_tables) > 0
        
        print(f"   ✓ Page {page_num}: Detected {len(validated_tables)} table(s)")
        return result
        
    except json.JSONDecodeError as e:
        print(f"   ✗ Page {page_num}: JSON decode error - {e}")
        print(f"   Response was: {response_text[:200]}...")
        return {"tables": [], "page_has_tables": False, "error": "json_decode_error"}
    except Exception as e:
        print(f"   ✗ Page {page_num}: Gemini API error - {e}")
        return {"tables": [], "page_has_tables": False, "error": str(e)}


# =============================================================================
# TABULA TABLE EXTRACTION
# =============================================================================

def extract_tables_with_tabula(file_path: str, page_num: int, bboxes: List[List[float]]) -> List[Dict]:
    """
    Extract tables from PDF using Tabula with detected bounding boxes.
    """
    extracted_tables = []
    
    for i, bbox in enumerate(bboxes):
        try:
            x1, y1, x2, y2 = bbox
            tabula_area = [y1, x1, y2, x2]  # Convert to Tabula format [top, left, bottom, right]
            
            # Try both modes
            tables_lattice = tabula.read_pdf(
                file_path, pages=page_num, area=tabula_area,
                lattice=True, multiple_tables=False, silent=True
            )
            
            tables_stream = tabula.read_pdf(
                file_path, pages=page_num, area=tabula_area,
                stream=True, multiple_tables=False, silent=True
            )
            
            # Choose best result
            table_df = None
            extraction_method = None
            
            if len(tables_lattice) > 0 and len(tables_stream) > 0:
                if tables_lattice[0].size > tables_stream[0].size:
                    table_df = tables_lattice[0]
                    extraction_method = "tabula_lattice"
                else:
                    table_df = tables_stream[0]
                    extraction_method = "tabula_stream"
            elif len(tables_lattice) > 0:
                table_df = tables_lattice[0]
                extraction_method = "tabula_lattice"
            elif len(tables_stream) > 0:
                table_df = tables_stream[0]
                extraction_method = "tabula_stream"
            
            if table_df is not None and not table_df.empty:
                table_markdown = table_df.to_markdown(index=False)
                table_text = table_df.to_string(index=False)
                
                extracted_tables.append({
                    "table_id": f"page{page_num}_table{i+1}",
                    "table_index": i,
                    "bbox": bbox,
                    "extraction_method": extraction_method,
                    "markdown": table_markdown,
                    "text": table_text,
                    "rows": len(table_df),
                    "columns": len(table_df.columns)
                })
                
                print(f"      ✓ Extracted table {i+1}: {len(table_df)} rows × {len(table_df.columns)} cols ({extraction_method})")
            else:
                print(f"      ✗ Table {i+1}: No data extracted")
                
        except Exception as e:
            print(f"      ✗ Table {i+1} extraction error: {e}")
            continue
    
    return extracted_tables


# =============================================================================
# TEXT EXTRACTION EXCLUDING TABLE REGIONS
# =============================================================================

def extract_text_excluding_tables(page: fitz.Page, table_bboxes: List[List[float]], 
                                   buffer_pixels: int = 5) -> List[Dict]:
    """
    Extract text from page excluding table regions.
    """
    text_blocks = []
    blocks = page.get_text("dict")["blocks"]
    
    for block in blocks:
        if block.get("type") == 0:  # Text block
            block_bbox = block["bbox"]
            block_text = ""
            
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    block_text += span.get("text", "") + " "
            
            block_text = block_text.strip()
            if not block_text:
                continue
            
            # Check overlap with tables
            overlaps_with_table = False
            for table_bbox in table_bboxes:
                tx1, ty1, tx2, ty2 = table_bbox
                bx0, by0, bx1, by1 = block_bbox
                
                # Add buffer
                tx1 -= buffer_pixels
                ty1 -= buffer_pixels
                tx2 += buffer_pixels
                ty2 += buffer_pixels
                
                # Check overlap
                if not (bx1 < tx1 or bx0 > tx2 or by1 < ty1 or by0 > ty2):
                    overlaps_with_table = True
                    break
            
            if not overlaps_with_table:
                text_blocks.append({
                    "text": block_text,
                    "bbox": list(block_bbox),
                    "position": block_bbox[1]
                })
    
    return text_blocks


# =============================================================================
# CONTENT MERGING IN READING ORDER
# =============================================================================

def merge_content_in_reading_order(tables: List[Dict], text_blocks: List[Dict], 
                                    page_num: int) -> List[Dict]:
    """
    Merge tables and text blocks in spatial reading order (top to bottom).
    """
    all_elements = []
    
    # Add tables
    for table in tables:
        all_elements.append({
            "type": "table",
            "content": f"--- {table['table_id']} ---\n\n{table['markdown']}\n\n---",
            "text": table['text'],
            "markdown": table['markdown'],
            "bbox": table['bbox'],
            "position": table['bbox'][1],
            "metadata": {
                "element_type": "table",
                "table_id": table['table_id'],
                "extraction_method": table['extraction_method'],
                "table_rows": table['rows'],
                "table_columns": table['columns']
            }
        })
    
    # Add text blocks
    for text_block in text_blocks:
        all_elements.append({
            "type": "text",
            "content": text_block['text'],
            "text": text_block['text'],
            "bbox": text_block['bbox'],
            "position": text_block['position'],
            "metadata": {
                "element_type": "text",
                "extraction_method": "pymupdf"
            }
        })
    
    # Sort by position
    all_elements.sort(key=lambda x: x['position'])
    
    # Add indices
    for i, element in enumerate(all_elements):
        element['metadata']['position_index'] = i
        element['metadata']['page_number'] = page_num
    
    return all_elements


# =============================================================================
# WORD COUNT VALIDATION
# =============================================================================

def validate_extraction(page: fitz.Page, extracted_elements: List[Dict], 
                        threshold: float = 0.90) -> Dict:
    """
    Validate extraction quality by comparing word counts.
    """
    total_text = page.get_text()
    total_words = len(total_text.split())
    
    extracted_text = " ".join([elem['text'] for elem in extracted_elements])
    extracted_words = len(extracted_text.split())
    
    ratio = extracted_words / total_words if total_words > 0 else 0
    passed = ratio >= threshold
    
    result = {
        "passed": passed,
        "total_words": total_words,
        "extracted_words": extracted_words,
        "ratio": ratio,
        "threshold": threshold
    }
    
    status = "✓ PASSED" if passed else "✗ FAILED"
    print(f"   Validation: {status} ({extracted_words}/{total_words} words, {ratio:.1%})")
    
    return result


print("✓ All helper functions loaded")

### 3. Main Processing Function

In [ ]:
def process_pdf_with_table_detection(file_path: str, dpi: int = 72) -> Dict:
    """
    Process a single PDF with table detection pipeline.
    
    This function:
    1. Checks if PDF is scanned (skips if yes)
    2. For each page:
       - Detects tables with Gemini Vision
       - Extracts tables with Tabula
       - Extracts non-table text with PyMuPDF
       - Merges in reading order
       - Validates extraction (90% threshold)
       - Falls back to OCR if validation fails
    3. Returns extracted content with comprehensive metadata
    
    Args:
        file_path: Path to PDF file
        dpi: DPI for page images (default: 150)
        
    Returns:
        Dictionary with extraction results
    """
    print(f"\n{'='*80}")
    print(f"Processing: {Path(file_path).name}")
    print(f"{'='*80}\n")
    
    # Check if scanned
    if is_scanned_pdf(file_path):
        print("\n⚠ PDF is scanned. Skipping (digital PDF processing only).\n")
        return {
            "file_name": Path(file_path).name,
            "status": "skipped",
            "reason": "scanned_pdf",
            "pages": []
        }
    
    # Open PDF
    doc = fitz.open(file_path)
    file_name = Path(file_path).name
    total_pages = len(doc)
    
    print(f"✓ Digital PDF detected. Processing {total_pages} pages...\n")
    
    all_page_data = []
    
    # Process each page
    for page_idx in range(total_pages):
        page_num = page_idx + 1
        print(f"\n--- Page {page_num}/{total_pages} ---")
        
        page = doc[page_idx]
        page_width = page.rect.width
        page_height = page.rect.height
        
        # Convert to image
        pix = page.get_pixmap(dpi=dpi)
        img_data = pix.tobytes("png")
        page_image = Image.open(io.BytesIO(img_data))
        
        # Detect tables
        print(f" → Detecting tables with Gemini Vision...")
        detection_result = detect_tables_with_gemini(page_image, page_width, page_height, page_num)
        table_bboxes = [table["bbox"] for table in detection_result.get("tables", [])]
        
        # Extract tables
        extracted_tables = []
        if table_bboxes:
            print(f" → Extracting {len(table_bboxes)} table(s) with Tabula...")
            extracted_tables = extract_tables_with_tabula(file_path, page_num, table_bboxes)
        
        # Extract text
        print(f" → Extracting text with PyMuPDF (excluding table regions)...")
        text_blocks = extract_text_excluding_tables(page, table_bboxes)
        print(f"   ✓ Extracted {len(text_blocks)} text block(s)")
        
        # Merge content
        print(f" → Merging content in reading order...")
        merged_elements = merge_content_in_reading_order(extracted_tables, text_blocks, page_num)
        print(f"   ✓ Merged {len(merged_elements)} elements")
        
        # Validate
        print(f" → Validating extraction...")
        validation = validate_extraction(page, merged_elements)
        
        # OCR fallback
        if not validation["passed"]:
            print(f" → Validation failed! Falling back to OCR...")
            try:
                ocr_text = pytesseract.image_to_string(page_image)
                if ocr_text.strip():
                    merged_elements = [{
                        "type": "text",
                        "content": ocr_text,
                        "text": ocr_text,
                        "bbox": [0, 0, page_width, page_height],
                        "position": 0,
                        "metadata": {
                            "element_type": "text",
                            "extraction_method": "ocr_fallback",
                            "page_number": page_num,
                            "position_index": 0,
                            "ocr_reason": "validation_failed"
                        }
                    }]
                    print(f"   ✓ OCR completed ({len(ocr_text.split())} words)")
            except Exception as e:
                print(f"   ✗ OCR failed: {e}")
        
        # Store page data
        page_data = {
            "page_number": page_num,
            "elements": merged_elements,
            "validation": validation,
            "tables_detected": len(extracted_tables),
            "text_blocks": len(text_blocks),
            "total_elements": len(merged_elements)
        }
        all_page_data.append(page_data)
    
    doc.close()
    
    # Summary
    total_tables = sum(p["tables_detected"] for p in all_page_data)
    total_elements = sum(p["total_elements"] for p in all_page_data)
    passed_validation = sum(1 for p in all_page_data if p["validation"]["passed"])
    
    print(f"\n{'='*80}")
    print(f"EXTRACTION SUMMARY")
    print(f"{'='*80}")
    print(f"File: {file_name}")
    print(f"Total Pages: {total_pages}")
    print(f"Total Tables Extracted: {total_tables}")
    print(f"Total Elements: {total_elements}")
    print(f"Validation: {passed_validation}/{total_pages} pages passed")
    print(f"{'='*80}\n")
    
    return {
        "file_name": file_name,
        "status": "completed",
        "total_pages": total_pages,
        "pages": all_page_data,
        "summary": {
            "total_tables": total_tables,
            "total_elements": total_elements,
            "pages_passed_validation": passed_validation
        }
    }


print("✓ Main processing function loaded")

### 4. Display Results Function

In [ ]:
def display_extraction_results(result: Dict, page_num: Optional[int] = None, 
                                max_text_length: int = 500):
    """
    Display extraction results in a readable format.
    
    Args:
        result: Extraction result dictionary
        page_num: Specific page to display (None = display all)
        max_text_length: Maximum text length to display per element
    """
    if result["status"] == "skipped":
        print(f"File: {result['file_name']}")
        print(f"Status: Skipped - {result['reason']}")
        return
    
    print(f"\n{'='*80}")
    print(f"EXTRACTION RESULTS: {result['file_name']}")
    print(f"{'='*80}\n")
    
    pages_to_display = result["pages"]
    if page_num is not None:
        pages_to_display = [p for p in pages_to_display if p["page_number"] == page_num]
    
    for page_data in pages_to_display:
        print(f"\n{'─'*80}")
        print(f"PAGE {page_data['page_number']}")
        print(f"{'─'*80}")
        print(f"Tables: {page_data['tables_detected']}")
        print(f"Text Blocks: {page_data['text_blocks']}")
        print(f"Total Elements: {page_data['total_elements']}")
        
        val = page_data['validation']
        print(f"Validation: {'✓ PASSED' if val['passed'] else '✗ FAILED'} "
              f"({val['extracted_words']}/{val['total_words']} words, {val['ratio']:.1%})")
        
        print(f"\nExtracted Content (in reading order):")
        print(f"{'─'*80}\n")
        
        for i, element in enumerate(page_data['elements']):
            elem_type = element['metadata']['element_type']
            method = element['metadata']['extraction_method']
            
            print(f"[{i+1}] Type: {elem_type.upper()} | Method: {method}")
            print(f"    Position Index: {element['metadata']['position_index']}")
            print(f"    BBox: {element['bbox']}")
            
            if elem_type == "table":
                table_id = element['metadata']['table_id']
                rows = element['metadata']['table_rows']
                cols = element['metadata']['table_columns']
                print(f"    Table ID: {table_id} ({rows} rows × {cols} cols)")
                print(f"    Content (Markdown):")
                markdown = element.get('markdown', element['content'])
                if len(markdown) > max_text_length:
                    print(f"    {markdown[:max_text_length]}...")
                else:
                    print(f"    {markdown}")
            else:
                text = element['text']
                if len(text) > max_text_length:
                    print(f"    Content: {text[:max_text_length]}...")
                else:
                    print(f"    Content: {text}")
            
            print()
    
    # Display summary
    if page_num is None:
        print(f"\n{'='*80}")
        print(f"SUMMARY")
        print(f"{'='*80}")
        print(f"Total Pages: {result['total_pages']}")
        print(f"Total Tables: {result['summary']['total_tables']}")
        print(f"Total Elements: {result['summary']['total_elements']}")
        print(f"Pages Passed Validation: {result['summary']['pages_passed_validation']}/{result['total_pages']}")
        print(f"{'='*80}\n")


print("✓ Display function loaded")

### 5. Test on Your PDF

In [ ]:
# =============================================================================
# CONFIGURE YOUR TEST FILE HERE
# =============================================================================

TEST_PDF_PATH = "./Documents/SFS XBRL PDF_53-54.pdf"  # CHANGE THIS

# Run extraction
result = process_pdf_with_table_detection(TEST_PDF_PATH)

# Store for inspection
extraction_result = result

### 6. View Results

In [ ]:
# Display all pages
display_extraction_results(extraction_result)

# Or display specific page:
# display_extraction_results(extraction_result, page_num=1)

### 7. Access Raw Data

You can access the extraction data programmatically:

In [ ]:
# Access specific page
page_1_data = extraction_result['pages'][0]
print(f"Page 1 has {page_1_data['tables_detected']} tables")

# Access specific element
first_element = page_1_data['elements'][0]
print(f"First element type: {first_element['metadata']['element_type']}")

# Get all tables across all pages
all_tables = []
for page in extraction_result['pages']:
    for element in page['elements']:
        if element['metadata']['element_type'] == 'table':
            all_tables.append(element)

print(f"Total tables found: {len(all_tables)}")